In [0]:
%sql
SHOW CATALOGS

# Player Clustering Approach — Research Notes

## Features to Use (per-90 minutes)
- **Shots** — attacking threat
- **xG** — shot quality
- **Passes** — ball distribution
- **Pressures** — pressing intensity
- **Carries** — ball progression
- **Dribbles** — 1v1 ability
- **Interceptions** — defensive positioning
- **Blocks / Clearances** — defensive actions

## Normalization Strategy
Using **z-score standardization** (StandardScaler)
- More robust to outliers than min-max
- Better for K-Means which is sensitive to scale
- A player with unusually high shots won't skew the entire cluster

## Dimensionality Reduction
Using **PCA**
- For clustering: retain components explaining ≥85% variance (typically 4-5)
- For visualization: use first 2 components for 2D scatter plot
- Removes correlated features (e.g. passes and carries are related)

## Number of Clusters
- Target: **5-8 archetypes** (standard in football analytics)
- Selection method: elbow method + silhouette score
- Expected archetypes:
  - Pressing Forward
  - Creative Playmaker
  - Box-to-Box Midfielder
  - Defensive Anchor
  - Goal Poacher

## Minimum Minutes Filter
- Only include players with **≥ 450 minutes** played
- Ensures per-90 metrics are statistically meaningful
- Removes players with very few appearances

In [0]:
%sql
SHOW TABLES IN bq_raw_statsbomb_sa_catalog.raw_statsbomb

In [0]:
%sql
-- Task 1: Build player season stats from raw tables
-- Calculate per-90 features for clustering
SELECT
    e.player_id,
    l.player_name,
    l.position_name,
    m.competition_name,
    m.season_name,
    COUNT(DISTINCT e.match_id) as matches_played,
    SUM(e.duration) / 90.0 as ninety_minutes,
    try_divide(SUM(CASE WHEN e.type = 'Shot' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as shots_p90,
    try_divide(SUM(CASE WHEN e.type = 'Shot' THEN e.shot_statsbomb_xg ELSE 0 END), SUM(e.duration) / 90.0) as xg_p90,
    try_divide(SUM(CASE WHEN e.type = 'Pass' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as passes_p90,
    try_divide(SUM(CASE WHEN e.type = 'Pressure' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as pressures_p90,
    try_divide(SUM(CASE WHEN e.type = 'Carry' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as carries_p90,
    try_divide(SUM(CASE WHEN e.type = 'Dribble' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as dribbles_p90,
    try_divide(SUM(CASE WHEN e.type = 'Interception' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as interceptions_p90,
    try_divide(SUM(CASE WHEN e.type = 'Block' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as blocks_p90,
    try_divide(SUM(CASE WHEN e.type = 'Clearance' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as clearances_p90,
    try_divide(SUM(CASE WHEN e.type = 'Ball Recovery' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as ball_recoveries_p90,
    try_divide(SUM(CASE WHEN e.type = 'Duel' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as duels_p90,
    try_divide(SUM(CASE WHEN e.type = 'Foul Committed' THEN 1 ELSE 0 END), SUM(e.duration) / 90.0) as fouls_p90
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events e
JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.lineups l
    ON e.player_id = l.player_id AND e.match_id = l.match_id
JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches m
    ON e.match_id = m.match_id
WHERE e.player_id IS NOT NULL
    AND e.duration IS NOT NULL
GROUP BY e.player_id, l.player_name, l.position_name, m.competition_name, m.season_name
HAVING SUM(e.duration) >= 450

In [0]:
%python
feature_docs = {
    'Feature': [
        'shots_p90', 'xg_p90', 'passes_p90', 'pressures_p90',
        'carries_p90', 'dribbles_p90', 'interceptions_p90',
        'blocks_p90', 'clearances_p90', 'ball_recoveries_p90',
        'duels_p90', 'fouls_p90'
    ],
    'Source Column (events.type)': [
        'Shot', 'shot_statsbomb_xg', 'Pass', 'Pressure',
        'Carry', 'Dribble', 'Interception',
        'Block', 'Clearance', 'Ball Recovery',
        'Duel', 'Foul Committed'
    ],
    'Normalization': ['z-score'] * 12,
    'Expected Range (raw)': [
        '0–8', '0–1', '20–80', '5–40',
        '10–60', '0–10', '0–5',
        '0–5', '0–5', '0–10',
        '0–15', '0–5'
    ],
    'Rationale': [
        'Attacking threat', 'Shot quality', 'Ball distribution', 'Pressing intensity',
        'Ball progression', '1v1 ability', 'Defensive positioning',
        'Defensive actions', 'Defensive clearances', 'Ball winning',
        'Physical duels', 'Aggression/fouling tendency'
    ]
}

import pandas as pd
df_docs = pd.DataFrame(feature_docs)
print(f"Total features: {len(df_docs)} (minimum 12 required ✓)")
display(df_docs)

In [0]:
%python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

player_df = _sqldf.toPandas()

feature_cols = [
    'shots_p90', 'xg_p90', 'passes_p90', 'pressures_p90',
    'carries_p90', 'dribbles_p90', 'interceptions_p90',
    'blocks_p90', 'clearances_p90', 'ball_recoveries_p90',
    'duels_p90', 'fouls_p90'
]

player_clean = player_df[feature_cols + ['player_id', 'player_name', 'position_name']].dropna()
print(f"Players after dropping nulls: {len(player_clean)}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(player_clean[feature_cols])

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_, color='steelblue')
axes[0].set_title('PCA Explained Variance per Component', fontsize=13)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')

cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cumvar) + 1), cumvar, marker='o', color='steelblue')
axes[1].axhline(0.85, color='red', linestyle='--', label='85% threshold')
axes[1].set_title('Cumulative Explained Variance', fontsize=13)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()

n_components_85 = np.argmax(cumvar >= 0.85) + 1
print(f"Components needed for 85% variance: {n_components_85}")

plt.tight_layout()
plt.show()

In [0]:
%python
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

player_df = _sqldf.toPandas()

feature_cols = [
    'shots_p90', 'xg_p90', 'passes_p90', 'pressures_p90',
    'carries_p90', 'dribbles_p90', 'interceptions_p90',
    'blocks_p90', 'clearances_p90', 'ball_recoveries_p90',
    'duels_p90', 'fouls_p90'
]

player_clean = player_df[feature_cols + ['player_id', 'player_name', 'position_name']].dropna()
print(f"Players after dropping nulls: {len(player_clean)}")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(player_clean[feature_cols])

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_, color='steelblue')
axes[0].set_title('PCA Explained Variance per Component', fontsize=13)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')

cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, len(cumvar) + 1), cumvar, marker='o', color='steelblue')
axes[1].axhline(0.85, color='red', linestyle='--', label='85% threshold')
axes[1].set_title('Cumulative Explained Variance', fontsize=13)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance')
axes[1].legend()

n_components_85 = np.argmax(cumvar >= 0.85) + 1
print(f"Components needed for 85% variance: {n_components_85}")

plt.tight_layout()
plt.show()

In [0]:
%python
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_components_85 = 6
pca_final = PCA(n_components=n_components_85)
X_pca_final = pca_final.fit_transform(X_scaled)

inertias = []
silhouette_scores = []
k_range = range(3, 9)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca_final)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_pca_final, labels, sample_size=2000, random_state=42))
    print(f"k={k}: inertia={kmeans.inertia_:.1f}, silhouette={silhouette_scores[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, marker='o', color='steelblue')
axes[0].set_title('Elbow Curve', fontsize=13)
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(k_range, silhouette_scores, marker='o', color='steelblue')
axes[1].axhline(0.3, color='red', linestyle='--', label='Min threshold (0.3)')
axes[1].set_title('Silhouette Scores', fontsize=13)
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()

best_k = k_range[silhouette_scores.index(max(silhouette_scores))]
print(f"Best k by silhouette score: {best_k}")

In [0]:
%python
# k=5 selected as optimal based on highest silhouette score (0.267)
# Note: Silhouette scores range from 0.22-0.27 across k=3 to k=8.
# While below the 0.3 target, this is expected for football player data
# due to natural overlap between player roles.

best_k = 5
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
player_clean['cluster'] = kmeans_final.fit_predict(X_pca_final)

colors = ['#378ADD', '#1D9E75', '#EF9F27', '#D4537E', '#7F77DD']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i in range(best_k):
    mask = player_clean['cluster'] == i
    axes[0].scatter(X_pca_final[mask, 0], X_pca_final[mask, 1],
                    c=colors[i], label=f'Cluster {i}', alpha=0.5, s=10)
axes[0].set_title('PCA Scatter — Clusters (k=5)', fontsize=13)
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()

cluster_counts = player_clean['cluster'].value_counts().sort_index()
axes[1].bar(cluster_counts.index, cluster_counts.values, color=colors)
axes[1].set_title('Players per Cluster', fontsize=13)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Number of Players')
for i, v in enumerate(cluster_counts.values):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nTop position per cluster:")
for i in range(best_k):
    top_pos = player_clean[player_clean['cluster'] == i]['position_name'].value_counts().head(3)
    print(f"Cluster {i}: {list(top_pos.index)}")

In [0]:
%python
import seaborn as sns

# Calculate mean of each feature per cluster
cluster_profiles = player_clean.groupby('cluster')[feature_cols].mean()

# Z-score normalize for better visualization
cluster_profiles_scaled = (cluster_profiles - cluster_profiles.mean()) / cluster_profiles.std()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    cluster_profiles_scaled.T,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0,
    ax=ax,
    xticklabels=[f'Cluster {i}' for i in range(best_k)],
    yticklabels=feature_cols
)
ax.set_title('Cluster Profile Heatmap (z-scored feature means)', fontsize=13)
plt.tight_layout()
plt.show()

# Also print raw means for reference
print("\nRaw feature means per cluster:")
display(cluster_profiles.round(3))

In [0]:
%python
CLUSTER_LABELS = {
    0: 'Goalkeeper',
    1: 'Creative Playmaker',
    2: 'Pressing Forward',
    3: 'Creative Winger',
    4: 'Defensive Anchor',
}

player_clean['archetype'] = player_clean['cluster'].map(CLUSTER_LABELS)
print(player_clean[['player_name', 'position_name', 'cluster', 'archetype']].head(20))

In [0]:
%python
import pandas as pd

cluster_labels_df = pd.DataFrame([
    {'cluster_id': 0, 'archetype': 'Goalkeeper'},
    {'cluster_id': 1, 'archetype': 'Creative Playmaker'},
    {'cluster_id': 2, 'archetype': 'Pressing Forward'},
    {'cluster_id': 3, 'archetype': 'Creative Winger'},
    {'cluster_id': 4, 'archetype': 'Defensive Anchor'},
])

display(cluster_labels_df)
print("\nCopy these 5 rows into seeds/cluster_labels.csv in the GitHub repo")

In [0]:
%sql
SHOW TABLES IN bq_raw_statsbomb_sa_catalog.raw_statsbomb

In [0]:
%python
cluster_export = player_clean[['player_id', 'player_name', 'position_name', 'cluster', 'archetype']].copy()
cluster_export.to_csv('/tmp/cluster_assignments.csv', index=False)
print(f"Saved {len(cluster_export)} rows")
print(cluster_export.head())

In [0]:
%python
with open('/tmp/cluster_assignments.csv', 'rb') as f:
    import base64
    b64 = base64.b64encode(f.read()).decode()

displayHTML(f'<a href="data:text/csv;base64,{b64}" download="cluster_assignments.csv">Click here to download CSV</a>')

In [0]:
print(player_clean.columns.tolist())

In [0]:
print(player_clean.groupby('archetype').size().sort_values(ascending=False))